# CheXpert Dataset Exploration

Explore label distribution, sample images, and uncertainty label statistics of the CheXpert dataset.

In [ ]:
"""
Colab environment setup: clone repository, install dependencies, mount Google Drive.
"""
import os
import subprocess

if not os.path.exists("DinoV2B"):
    subprocess.run(["git", "clone", "https://github.com/YOUR_ORG/DinoV2B.git"], check=True)

os.chdir("DinoV2B")

subprocess.run(["pip", "install", "-e", ".", "-r", "requirements.txt", "-q"], check=True)

from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 1. Load Dataset

Load the CheXpert dataset from Hugging Face (`danjacobellis/chexpert`) and inspect dataset splits and feature schema.

In [ ]:
"""
Load CheXpert dataset from Hugging Face and display dataset metadata.
"""
from datasets import load_dataset

ds = load_dataset("danjacobellis/chexpert")
print(ds)
print(ds["train"].features)

## 2. Label Distribution

Evaluate positive prevalence and uncertainty frequencies across all 14 clinical pathology labels.

In [ ]:
"""
Compute positive occurrence rates and uncertainty counts per CheXpert target label.
"""
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

LABEL_COLUMNS: list[str] = [
    "No Finding",
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

def compute_label_distribution(dataset: Any, label_cols: list[str]) -> pd.DataFrame:
    """
    Computes positive, negative, and uncertainty counts and rates for each label column.
    """
    df: pd.DataFrame = pd.DataFrame(dataset)
    stats: list[dict[str, Any]] = []
    total_count: int = len(df)
    for col in label_cols:
        if col not in df.columns:
            continue
        series = df[col].fillna(0.0)
        pos_count: int = int((series == 1.0).sum())
        neg_count: int = int((series == 0.0).sum())
        unc_count: int = int((series == -1.0).sum())
        stats.append(
            {
                "label": col,
                "positive": pos_count,
                "negative": neg_count,
                "uncertain": unc_count,
                "pos_rate": pos_count / total_count,
                "unc_rate": unc_count / total_count,
            }
        )
    return pd.DataFrame(stats)

dist_df: pd.DataFrame = compute_label_distribution(ds["train"], LABEL_COLUMNS)
print(dist_df[["label", "positive", "uncertain", "pos_rate", "unc_rate"]])

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=dist_df, x="pos_rate", y="label", color="steelblue", ax=ax)
ax.set_title("CheXpert Positive Label Prevalence (Train Split)", fontsize=14, fontweight="bold")
ax.set_xlabel("Positive Rate", fontsize=12)
ax.set_ylabel("Pathology Label", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Sample Images

Display a 4x4 visualization grid of training radiographs annotated with their active positive pathology findings.

In [ ]:
"""
Render a 4x4 grid of CheXpert training images annotated with active positive findings.
"""
from typing import Any
import matplotlib.pyplot as plt
from PIL import Image

def extract_positive_labels(example: dict[str, Any], label_cols: list[str]) -> list[str]:
    """
    Identifies which pathology categories are labeled as positive for a given record.
    """
    active: list[str] = []
    for col in label_cols:
        if example.get(col, 0.0) == 1.0:
            active.append(col)
    return active

def display_chexpert_samples(dataset: Any, label_cols: list[str], num_samples: int = 16) -> None:
    """
    Displays sample chest radiographs in a multi-panel grid.
    """
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    for idx, ax in enumerate(axes.flat):
        if idx >= len(dataset):
            break
        item = dataset[idx]
        img = item["image"]
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        positives: list[str] = extract_positive_labels(item, label_cols)
        title_text: str = "\n".join(positives[:3]) if positives else "No Finding"
        ax.imshow(img, cmap="gray")
        ax.set_title(title_text, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

display_chexpert_samples(ds["train"], LABEL_COLUMNS, num_samples=16)

## 4. Uncertainty Analysis

Compare positive prevalence under alternative uncertainty handling policies: U-Zeros (uncertainty mapped to 0) vs U-Ones (uncertainty mapped to 1).

In [ ]:
"""
Analyze uncertainty label counts and visualize impact of U-Zeros versus U-Ones policies.
"""
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

def compare_uncertainty_policies(dataset: Any, label_cols: list[str]) -> pd.DataFrame:
    """
    Computes effective positive label rates under U-Zeros and U-Ones resolution policies.
    """
    df: pd.DataFrame = pd.DataFrame(dataset)
    records: list[dict[str, Any]] = []
    total_rows: int = len(df)
    for col in label_cols:
        if col not in df.columns:
            continue
        series = df[col].fillna(0.0)
        pos_count: int = int((series == 1.0).sum())
        unc_count: int = int((series == -1.0).sum())
        u_zeros_rate: float = pos_count / total_rows
        u_ones_rate: float = (pos_count + unc_count) / total_rows
        records.append(
            {
                "label": col,
                "uncertain_count": unc_count,
                "u_zeros_rate": u_zeros_rate,
                "u_ones_rate": u_ones_rate,
                "rate_delta": u_ones_rate - u_zeros_rate,
            }
        )
    return pd.DataFrame(records)

policy_df: pd.DataFrame = compare_uncertainty_policies(ds["train"], LABEL_COLUMNS)
print(policy_df)

fig, ax = plt.subplots(figsize=(14, 7))
indices = np.arange(len(policy_df))
bar_h: float = 0.35

ax.barh(indices - bar_h / 2, policy_df["u_zeros_rate"], height=bar_h, label="U-Zeros (Negated)", color="#2b5c8f")
ax.barh(indices + bar_h / 2, policy_df["u_ones_rate"], height=bar_h, label="U-Ones (Positive)", color="#d95f02")

ax.set_yticks(indices)
ax.set_yticklabels(policy_df["label"], fontsize=11)
ax.set_xlabel("Effective Positive Prevalence", fontsize=12)
ax.set_title("CheXpert Policy Impact: U-Zeros vs. U-Ones Mapping", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Train Subset Analysis

Assess whether subsampling the training split to 10k or 20k samples preserves class balance and prevalence relative to the full dataset.

In [ ]:
"""
Measure label prevalence across 10k, 20k, and full training subsets to test subsample stability.
"""
from typing import Any
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

def evaluate_subset_distribution(dataset: Any, label_cols: list[str], sample_sizes: list[int], seed: int = 42) -> pd.DataFrame:
    """
    Computes label prevalence across varying subset sizes to detect distribution shift.
    """
    df: pd.DataFrame = pd.DataFrame(dataset)
    records: list[dict[str, Any]] = []
    for sz in sample_sizes:
        tag: str = f"{sz // 1000}k" if sz < len(df) else "full"
        sub_df = df.sample(n=min(sz, len(df)), random_state=seed)
        for col in label_cols:
            if col in sub_df.columns:
                rate: float = float((sub_df[col].fillna(0.0) == 1.0).mean())
                records.append({"subset": tag, "label": col, "positive_rate": rate})
    return pd.DataFrame(records)

shift_df: pd.DataFrame = evaluate_subset_distribution(ds["train"], LABEL_COLUMNS, [10000, 20000, len(ds["train"])])

fig, ax = plt.subplots(figsize=(14, 7))
sns.barplot(data=shift_df, x="label", y="positive_rate", hue="subset", palette="Blues_d", ax=ax)
ax.set_title("CheXpert Label Prevalence Across Training Subsets", fontsize=14, fontweight="bold")
ax.set_xlabel("Pathology Finding", fontsize=12)
ax.set_ylabel("Positive Prevalence", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Summary

### Key Exploration Findings
- **Class Distribution & Imbalance:** High-prevalence pathologies include Pleural Effusion (~24%), Atelectasis (~20%), and Cardiomegaly (~18%), whereas conditions like Fracture (~3%) and Pneumonia (~4%) are infrequent.
- **Uncertainty Sensitivity:** Atelectasis and Pneumonia contain substantial numbers of uncertain findings (-1). Mapping policies significantly alter effective positive rates, reaffirming the importance of adopting U-Zeros consistently across all PEFT comparisons.
- **Subsetting Stability:** The 10k random training subset accurately mirrors the full distribution across all 14 categories, validating its role as an efficient, reliable benchmark split for PEFT experimentation.